In [ ]:
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from datasets import load_dataset, Audio
import torch
import numpy as np
import IPython.display
from tqdm.auto import tqdm

In [ ]:
yodas2 = (
    load_dataset('yodas2_ru000_16k', split='train', data_dir='audio')
    .cast_column('audio', Audio(sampling_rate=16_000))
)

In [ ]:
sample = yodas2[15]
waveform = sample['audio']['array']
rate = sample['audio']['sampling_rate']
assert len(waveform) / rate < 300
IPython.display.Audio(waveform, rate=rate)

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
model = ASTForAudioClassification.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
labels = np.array([model.config.id2label[id] for id in range(model.config.num_labels)])

In [ ]:
span = 10
shift = 10

scores_for_segments = []

for segment_idx in tqdm(np.arange(0, len(waveform) / rate - span, step=shift)):
    segment = waveform[int(segment_idx * rate) : int((segment_idx + span) * rate)]

    segment = np.tile(segment[None], (64, 1))

    # audio file is decoded on the fly
    inputs = feature_extractor(segment, sampling_rate=16_000, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    scores = logits[0].detach().cpu().numpy()

    scores_for_segments.append(scores)

scores_for_segments = np.array(scores_for_segments)

In [ ]:
import matplotlib.pyplot as plt

classes_to_display = np.argsort(scores_for_segments.max(axis=0))[:-10:-1]
classes_to_display

for cls_idx in classes_to_display:
    plt.plot(scores_for_segments[:, cls_idx], label=labels[cls_idx])

plt.legend()
plt.show()

In [ ]:
plt.matshow(scores_for_segments[:, classes_to_display].T, aspect='auto')